# Heart Disease Prediction - MLflow Experiment Tracking

This notebook demonstrates experiment tracking using MLflow. The objective is to log model parameters, evaluation metrics, artifacts, and trained models to ensure reproducibility and facilitate comparison between different machine learning experiments.

In [10]:
# Import Required Libraries

import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import mlflow
import mlflow.sklearn

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

In [11]:
# Load Dataset

heart_df = pd.read_csv("../data/processed/heart.csv")

heart_df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [12]:
# Feature and Target

X = heart_df.drop("target", axis=1)

y = heart_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [13]:
# Load Saved Model

model = joblib.load("../models/heart_disease_model.pkl")

print(model)

RandomForestClassifier(max_depth=5, min_samples_split=5, random_state=42)


In [14]:
# Predictions

predictions = model.predict(X_test)

probabilities = model.predict_proba(X_test)[:,1]

In [15]:
# Evaluation Metrics

accuracy = accuracy_score(y_test, predictions)

precision = precision_score(y_test, predictions)

recall = recall_score(y_test, predictions)

f1 = f1_score(y_test, predictions)

auc = roc_auc_score(y_test, probabilities)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", auc)

Accuracy : 0.9016393442622951
Precision: 0.8666666666666667
Recall   : 0.9285714285714286
F1 Score : 0.896551724137931
ROC-AUC  : 0.948051948051948


## MLflow Experiment Tracking

The trained model is tracked using MLflow. Model parameters, evaluation metrics, artifacts, and the trained model are logged for reproducibility and comparison.

In [37]:
import os

# Allow MLflow to use the local file store
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import mlflow

In [39]:
import os
import mlflow

# Set MLflow tracking directory to project root
tracking_path = os.path.abspath("../mlruns")

mlflow.set_tracking_uri(f"file:///{tracking_path.replace(os.sep, '/')}")

print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:///c:/Projects/Heart-Disease-MLOps/mlruns


In [18]:
# Create MLflow Experiment

mlflow.set_experiment("Heart Disease Prediction")

<Experiment: artifact_location='file:///c:/Projects/Heart-Disease-MLOps/mlruns/888722970509823065', creation_time=1782922114438, effective_trace_archival_retention=None, experiment_id='888722970509823065', last_update_time=1782922114438, lifecycle_stage='active', name='Heart Disease Prediction', tags={}, trace_location=None, workspace='default'>

In [19]:
# MLflow Run

with mlflow.start_run(run_name="Tuned Random Forest"):

    # Log Parameters
    mlflow.log_param("Model", "Random Forest")

    mlflow.log_param("n_estimators", model.n_estimators)
    mlflow.log_param("max_depth", model.max_depth)
    mlflow.log_param("min_samples_split", model.min_samples_split)
    mlflow.log_param("min_samples_leaf", model.min_samples_leaf)

    # Log Metrics
    mlflow.log_metric("Accuracy", accuracy)
    mlflow.log_metric("Precision", precision)
    mlflow.log_metric("Recall", recall)
    mlflow.log_metric("F1 Score", f1)
    mlflow.log_metric("ROC-AUC", auc)

    print("Parameters and Metrics Logged Successfully!")

Parameters and Metrics Logged Successfully!


## Confusion Matrix

The confusion matrix is generated and logged as an MLflow artifact to visualize the classification performance of the trained model.

In [20]:
# Confusion Matrix

os.makedirs("../reports/figures", exist_ok=True)

fig, ax = plt.subplots(figsize=(6,5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    cmap="Blues",
    ax=ax
)

plt.title("Confusion Matrix")

confusion_matrix_path = "../reports/figures/confusion_matrix.png"

plt.savefig(confusion_matrix_path)

plt.close()

## ROC Curve

The ROC Curve illustrates the model's ability to distinguish between positive and negative classes. The plot is stored as an MLflow artifact.

In [21]:
# ROC Curve

fig, ax = plt.subplots(figsize=(6,5))

RocCurveDisplay.from_predictions(
    y_test,
    probabilities,
    ax=ax
)

plt.title("ROC Curve")

roc_curve_path = "../reports/figures/roc_curve.png"

plt.savefig(roc_curve_path)

plt.close()

In [22]:
# Log Artifacts and Model

with mlflow.start_run(run_name="Tuned Random Forest Artifacts"):

    # Parameters
    mlflow.log_param("Model", "Random Forest")
    mlflow.log_param("n_estimators", model.n_estimators)
    mlflow.log_param("max_depth", model.max_depth)
    mlflow.log_param("min_samples_split", model.min_samples_split)
    mlflow.log_param("min_samples_leaf", model.min_samples_leaf)

    # Metrics
    mlflow.log_metric("Accuracy", accuracy)
    mlflow.log_metric("Precision", precision)
    mlflow.log_metric("Recall", recall)
    mlflow.log_metric("F1 Score", f1)
    mlflow.log_metric("ROC-AUC", auc)

    # Artifacts
    mlflow.log_artifact(confusion_matrix_path)
    mlflow.log_artifact(roc_curve_path)

    # Save Model
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="heart_disease_model"
    )

print("Model and artifacts logged successfully!")

2026/07/01 21:41:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Model and artifacts logged successfully!


In [31]:
# Verify MLflow Directory

print("Files inside project directory:")

print(os.listdir("../"))

Files inside project directory:
['.github', '.gitignore', 'data', 'docker', 'Dockerfile', 'k8s', 'miruns', 'mlflow.db', 'mlruns', 'models', 'notebooks', 'README.md', 'reports', 'requirements.txt', 'screenshots', 'src', 'tests', 'venv']


In [34]:
import mlflow

print(mlflow.get_tracking_uri())

file:///c:/Projects/Heart-Disease-MLOps/mlruns


In [35]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiments = client.search_experiments()

for exp in experiments:
    print(f"Experiment: {exp.name}")
    runs = client.search_runs([exp.experiment_id])
    print(f"Number of runs: {len(runs)}")

Experiment: Heart Disease Prediction
Number of runs: 4
Experiment: Default
Number of runs: 0
